In [1]:

# If needed, uncomment these:
# %pip install pandas geopandas pyproj shapely scikit-learn tqdm

import os
import re
import glob
import math
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
from sklearn.neighbors import BallTree
from tqdm import tqdm
tqdm.pandas()


In [24]:
# Project-relative paths (adjust if different)
DATA_DIR = "data"
OUT_DIR  = "out"
os.makedirs(OUT_DIR, exist_ok=True)

# Your files
ZCTA_DIR = os.path.join(DATA_DIR, "tl_2020_us_zcta510")
RUCA_CSV = os.path.join(DATA_DIR, "RUCA-codes-2020-zipcode.csv")

# Trial files (all *_with_zip.csv)
TRIAL_FILES = sorted(glob.glob(os.path.join(DATA_DIR, "*_with_zip.csv")))

# If you later download ACS/NHGIS population by ZCTA, set the path here:
ACS_POP_FILE = None  # e.g., "../data/nhgis/acs_zcta_population.csv"

# Column names that might contain a ZIP (we'll search in this list)
ZIP_COLS = ["zip", "zipcode", "ZIP", "ZIPCode"]

# --- helpers ---
ZIP_RE = re.compile(r"^\d{5}$")

def to_zip5(x):
    """Return 5-digit ZIP string or None."""
    if pd.isna(x):
        return None
    s = str(x).strip()
    # strip 4+4 formats or text
    s = re.sub(r"[^\d]", "", s)
    if len(s) >= 5:
        s = s[:5]
    return s if ZIP_RE.match(s) else None

def classify_rural_urban(primary_ruca):
    """
    RUCA: typical cut:
      Urban: 1.0–3.0
      Rural: 4.0–10.0
    Returns 'Urban' or 'Rural' (None if missing)
    """
    try:
        p = float(primary_ruca)
    except:
        return None
    return "Urban" if 1 <= p < 4 else "Rural"


In [25]:
# Load 2020 ZCTA polygons
zcta = gpd.read_file(ZCTA_DIR)

# Typical id column names in tl_2020_us_zcta510:
# 'ZCTA5CE10' (5-digit ZCTA code), geometry
id_col = "ZCTA5CE10" if "ZCTA5CE10" in zcta.columns else None
assert id_col is not None, f"Could not find ZCTA id col in {list(zcta.columns)}"

# Ensure CRS is WGS84 and compute centroids
if zcta.crs is None or zcta.crs.to_epsg() != 4269:
    # These TIGER files are usually EPSG:4269 (NAD83); we’ll project to 4326 for lat/lon
    pass

zcta = zcta.to_crs(epsg=4326)
zcta["centroid"] = zcta.geometry.centroid
zcta["lon"] = zcta["centroid"].x
zcta["lat"] = zcta["centroid"].y

zip_centroids = zcta[[id_col, "lat", "lon"]].rename(columns={id_col: "zip"})
# make sure zip is 5-digit string
zip_centroids["zip"] = zip_centroids["zip"].astype(str).str.zfill(5)

print(zip_centroids.head())
print(f"ZCTA count: {len(zip_centroids):,}")


/var/folders/4l/64tth9h948n3dbq1py15y2qr0000gn/T/ipykernel_76260/2670556587.py:15: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  zcta["centroid"] = zcta.geometry.centroid


     zip        lat        lon
0  43451  41.312827 -83.615185
1  43452  41.515177 -82.966801
2  43456  41.668369 -82.822768
3  43457  41.267330 -83.427484
4  43458  41.531051 -83.212587
ZCTA count: 33,144


In [26]:
ruca = pd.read_csv(RUCA_CSV, dtype=str)
# Your sample columns looked like: ZIPCode, State, ZIPCodeType, POName, PrimaryRUCA, SecondaryRUCA
# Standardize
ruca["zip"] = ruca["ZIPCode"].apply(to_zip5)
ruca["PrimaryRUCA_num"] = pd.to_numeric(ruca["PrimaryRUCA"], errors="coerce")
ruca["rural_urban"] = ruca["PrimaryRUCA_num"].apply(classify_rural_urban)

ruca = ruca[["zip", "PrimaryRUCA", "rural_urban"]].dropna(subset=["zip"]).drop_duplicates()
print(ruca["rural_urban"].value_counts(dropna=False))


rural_urban
Urban    23381
Rural    17765
Name: count, dtype: int64


In [27]:
# 4) Load & combine all GI trial site CSVs

ZIP_COLS = ["zip", "zipcode", "Zipcode", "ZIP", "ZIPCode", "Zip", "postal_code"]

ZIP_RE = re.compile(r"\b\d{5}\b")

def extract_zips_from_locations(text):
    if pd.isna(text):
        return []
    s = str(text)
    # Find all 5-digit substrings (handles things like "... MA 02115; Houston, TX 77030")
    zips = ZIP_RE.findall(s)
    # Deduplicate while preserving order
    seen = set()
    out = []
    for z in zips:
        if z not in seen:
            seen.add(z)
            out.append(z)
    return out

def load_trials_one(path):
    df = pd.read_csv(path, dtype=str)
    
    # Try to find an explicit ZIP column first
    zip_col = None
    for c in ZIP_COLS:
        if c in df.columns:
            zip_col = c
            break

    if zip_col is not None:
        df["zip"] = df[zip_col].apply(to_zip5)
        df = df.dropna(subset=["zip"]).copy()
    elif "Locations" in df.columns:
        # Fallback: parse from Locations text (explode to one row per ZIP)
        df["__zips_list"] = df["Locations"].apply(extract_zips_from_locations)
        df = df.explode("__zips_list")
        df["zip"] = df["__zips_list"].apply(to_zip5)
        df = df.dropna(subset=["zip"]).drop(columns="__zips_list")
    else:
        raise ValueError(
            f"No ZIP column and no 'Locations' field to parse in {path}. "
            f"Available columns: {list(df.columns)}"
        )

    # Infer cancer type from filename if not present
    if "cancer_type" not in df.columns:
        fname = os.path.basename(path).lower()
        cancer = re.sub(r"_with_zip.*$", "", fname).replace(".csv", "")
        df["cancer_type"] = cancer

    # Keep only what we need
    keep_cols = ["NCT Number","Study Title","Conditions","Interventions","Sponsor","Locations"]
    for col in keep_cols:
        if col not in df.columns:
            df[col] = np.nan

    return df[keep_cols + ["zip","cancer_type"]]

trial_list = []
for f in TRIAL_FILES:
    try:
        trial_list.append(load_trials_one(f))
        print(f"Loaded: {os.path.basename(f)} ({len(trial_list[-1]):,} rows after ZIP parsing)")
    except Exception as e:
        print(f"Skipping {f}: {e}")

trials = (
    pd.concat(trial_list, ignore_index=True)
    if trial_list else
    pd.DataFrame(columns=["zip","cancer_type"])
)

# Dedup by trial site ZIP within each cancer type
trials = trials.drop_duplicates(subset=["zip","cancer_type"]).reset_index(drop=True)

print(trials.head())
print(f"Unique trial ZIP entries: {len(trials):,} across {trials['cancer_type'].nunique()} cancer types")


Loaded: Oesophageal_cancer_with_zip.csv (3,090 rows after ZIP parsing)
Loaded: cholangiocarcinoma_with_zip.csv (774 rows after ZIP parsing)
Loaded: colorectal_cancer_with_zip.csv (6,396 rows after ZIP parsing)
Loaded: gastric_cancer_with_zip.csv (5,287 rows after ZIP parsing)
Loaded: hepatocellular_cancer_with_zip.csv (1,123 rows after ZIP parsing)
Loaded: pancreatic_cancer_with_zip.csv (5,441 rows after ZIP parsing)
    NCT Number                                        Study Title  \
0  NCT01684904               Proton Therapy for Esophageal Cancer   
1  NCT06872515  Impact of Prehabilitation in Oncology Via Exer...   
2  NCT06906887  PULSO Trial: Pulsed Low-Dose-Rate (PLDR) Radia...   
3  NCT06430346  Exercise Prehabilitation for Locoregional Esop...   
4  NCT06414902  18F-AraG PET/CT as a Non-Invasive Imaging Biom...   

                                          Conditions  \
0                                   Esophagus Cancer   
1             Esophageal Cancer|Esophageal Carcinoma

In [28]:
trial_sites = trials.merge(zip_centroids, on="zip", how="left")
missing_geo = trial_sites["lat"].isna().sum()
if missing_geo:
    print(f"WARNING: {missing_geo} trial rows lack ZCTA centroid match. They will be dropped for distance calcs.")
trial_sites = trial_sites.dropna(subset=["lat","lon"]).copy()

# Build one per ZIP per cancer type (we already deduped), keep site coordinates
trial_sites = trial_sites[["zip","cancer_type","lat","lon"]].drop_duplicates()
print(trial_sites.head())
print(f"Geocoded trial site rows: {len(trial_sites):,}")


     zip         cancer_type        lat         lon
0  92354  oesophageal_cancer  34.049773 -117.251508
1  15213  oesophageal_cancer  40.444041  -79.955247
2  53226  oesophageal_cancer  43.049438  -88.042176
3  33612  oesophageal_cancer  28.050741  -82.449746
4  77030  oesophageal_cancer  29.706274  -95.402594
Geocoded trial site rows: 5,912


In [29]:
zip_universe = zip_centroids.merge(ruca, on="zip", how="left")

# Optional: attach population if you have ACS later
if ACS_POP_FILE and os.path.exists(ACS_POP_FILE):
    pop = pd.read_csv(ACS_POP_FILE, dtype={ "zip": str })
    pop["zip"] = pop["zip"].apply(to_zip5)
    pop = pop.dropna(subset=["zip"])
    # Expect a 'population' column in ACS file
    if "population" in pop.columns:
        zip_universe = zip_universe.merge(pop[["zip","population"]], on="zip", how="left")
    else:
        print("ACS file found but no 'population' column detected; skipping population merge.")
else:
    zip_universe["population"] = np.nan  # placeholder

# Keep only ZIPs with coordinates and RUCA classification for coverage stats
zip_universe = zip_universe.dropna(subset=["lat","lon"])
print(zip_universe.head())
print(f"Zip universe rows: {len(zip_universe):,}")


     zip        lat        lon PrimaryRUCA rural_urban  population
0  43451  41.312827 -83.615185           2       Urban         NaN
1  43452  41.515177 -82.966801           1       Urban         NaN
2  43456  41.668369 -82.822768          10       Rural         NaN
3  43457  41.267330 -83.427484           3       Urban         NaN
4  43458  41.531051 -83.212587           2       Urban         NaN
Zip universe rows: 33,144


In [30]:
# Build a BallTree (haversine) for each cancer type and for "any GI trial"
EARTH_RADIUS_MILES = 3958.7613
RADIUS_MILES = 50.0
radius_radians = RADIUS_MILES / EARTH_RADIUS_MILES

def build_tree(df):
    # expects columns: lat, lon in degrees
    pts = np.deg2rad(df[["lat","lon"]].values)
    return BallTree(pts, metric="haversine"), pts

# "Any GI" tree
any_tree, any_pts = build_tree(trial_sites)

# Cancer-specific trees
trees_by_cancer = {}
for cancer, dfc in trial_sites.groupby("cancer_type"):
    trees_by_cancer[cancer] = build_tree(dfc)

# Now query coverage for each ZIP centroid
query_pts = np.deg2rad(zip_universe[["lat","lon"]].values)

# Any GI within 50 miles?
ind_any = any_tree.query_radius(query_pts, r=radius_radians)
zip_universe["covered_any_gi_50mi"] = [1 if len(ix)>0 else 0 for ix in ind_any]

# Cancer-specific coverage flags
for cancer, (tree, _) in trees_by_cancer.items():
    inds = tree.query_radius(query_pts, r=radius_radians)
    flag = [1 if len(ix)>0 else 0 for ix in inds]
    col = f"covered_{cancer}_50mi"
    # sanitize col name (just in case)
    col = re.sub(r"[^a-z0-9_]+","_", col.lower())
    zip_universe[col] = flag

zip_universe.head()


,zip,lat,lon,PrimaryRUCA,rural_urban,population,covered_any_gi_50mi,covered_cholangiocarcinoma_50mi,covered_colorectal_cancer_50mi,covered_gastric_cancer_50mi,covered_hepatocellular_cancer_50mi,covered_oesophageal_cancer_50mi,covered_pancreatic_cancer_50mi
0,43451,41.312827,-83.615185,2,Urban,NaN,1,0,1,1,1,1,1
1,43452,41.515177,-82.966801,1,Urban,NaN,1,0,1,1,1,1,1
2,43456,41.668369,-82.822768,10,Rural,NaN,1,1,1,1,1,1,1
3,43457,41.267330,-83.427484,3,Urban,NaN,1,0,1,1,1,1,1
4,43458,41.531051,-83.212587,2,Urban,NaN,1,0,1,1,1,1,1


In [31]:
def summarize_coverage(df, cover_col, weight_col=None):
    # Drop rows with unknown RUCA classification
    dfx = df[df["rural_urban"].isin(["Urban","Rural"])].copy()
    if not len(dfx):
        return pd.DataFrame()

    if weight_col and weight_col in dfx.columns and dfx[weight_col].notna().any():
        # Weighted share = sum(w * covered) / sum(w)
        grp = dfx.groupby("rural_urban").apply(
            lambda g: pd.Series({
                "n_zips": len(g),
                "share_covered_unweighted": g[cover_col].mean(),
                "pop_covered": (g[cover_col] * g[weight_col]).sum(skipna=True),
                "pop_total": g[weight_col].sum(skipna=True)
            })
        ).reset_index()
        grp["share_covered_weighted"] = grp["pop_covered"] / grp["pop_total"]
        grp.drop(columns=["pop_covered","pop_total"], inplace=True)
    else:
        grp = dfx.groupby("rural_urban").agg(
            n_zips=("zip","count"),
            share_covered_unweighted=(cover_col,"mean")
        ).reset_index()

    grp["coverage_flag"] = cover_col
    return grp

# Any GI coverage summary
summary_any = summarize_coverage(zip_universe, "covered_any_gi_50mi", weight_col="population")

# Per-cancer summaries
cancer_cols = [c for c in zip_universe.columns if c.startswith("covered_") and c.endswith("_50mi") and c != "covered_any_gi_50mi"]
summary_cancers = pd.concat(
    [summarize_coverage(zip_universe, c, weight_col="population") for c in cancer_cols],
    ignore_index=True
) if cancer_cols else pd.DataFrame()

summary_any, summary_cancers.head()


(  rural_urban  n_zips  share_covered_unweighted        coverage_flag
 0       Rural   16160                  0.773515  covered_any_gi_50mi
 1       Urban   16960                  0.971226  covered_any_gi_50mi,
   rural_urban  n_zips  share_covered_unweighted  \
 0       Rural   16160                  0.337809   
 1       Urban   16960                  0.748054   
 2       Rural   16160                  0.718750   
 3       Urban   16960                  0.957075   
 4       Rural   16160                  0.701238   
 
                      coverage_flag  
 0  covered_cholangiocarcinoma_50mi  
 1  covered_cholangiocarcinoma_50mi  
 2   covered_colorectal_cancer_50mi  
 3   covered_colorectal_cancer_50mi  
 4      covered_gastric_cancer_50mi  )

In [37]:
zip_universe.head()

,zip,lat,lon,PrimaryRUCA,rural_urban,population,covered_any_gi_50mi,covered_cholangiocarcinoma_50mi,covered_colorectal_cancer_50mi,covered_gastric_cancer_50mi,covered_hepatocellular_cancer_50mi,covered_oesophageal_cancer_50mi,covered_pancreatic_cancer_50mi
0,43451,41.312827,-83.615185,2,Urban,NaN,1,0,1,1,1,1,1
1,43452,41.515177,-82.966801,1,Urban,NaN,1,0,1,1,1,1,1
2,43456,41.668369,-82.822768,10,Rural,NaN,1,1,1,1,1,1,1
3,43457,41.267330,-83.427484,3,Urban,NaN,1,0,1,1,1,1,1
4,43458,41.531051,-83.212587,2,Urban,NaN,1,0,1,1,1,1,1


In [33]:
print("\nAny GI trials within 50 miles:")
print(summary_any)

if not summary_cancers.empty:
    print("\nTop 10 cancer-specific coverage flags:")
    display(summary_cancers.head(10))



Any GI trials within 50 miles:
  rural_urban  n_zips  share_covered_unweighted        coverage_flag
0       Rural   16160                  0.773515  covered_any_gi_50mi
1       Urban   16960                  0.971226  covered_any_gi_50mi

Top 10 cancer-specific coverage flags:


,rural_urban,n_zips,share_covered_unweighted,coverage_flag
0,Rural,16160,0.337809,covered_cholangiocarcinoma_50mi
1,Urban,16960,0.748054,covered_cholangiocarcinoma_50mi
2,Rural,16160,0.718750,covered_colorectal_cancer_50mi
3,Urban,16960,0.957075,covered_colorectal_cancer_50mi
4,Rural,16160,0.701238,covered_gastric_cancer_50mi
5,Urban,16960,0.943101,covered_gastric_cancer_50mi
6,Rural,16160,0.330631,covered_hepatocellular_cancer_50mi
7,Urban,16960,0.768337,covered_hepatocellular_cancer_50mi
8,Rural,16160,0.624443,covered_oesophageal_cancer_50mi
9,Urban,16960,0.897465,covered_oesophageal_cancer_50mi


In [34]:
# QC: counts & missing RUCA
print("ZIP universe rows:", len(zip_universe))
print(zip_universe["rural_urban"].value_counts(dropna=False))

missing_ruca = zip_universe["rural_urban"].isna().sum()
if missing_ruca:
    print(f"WARNING: {missing_ruca} ZIPs have missing RUCA; excluded from summaries.")


ZIP universe rows: 33144
rural_urban
Urban    16960
Rural    16160
NaN         24
Name: count, dtype: int64


In [35]:
import math
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

def two_group_stats(df, cover_col):
    dfx = df[df["rural_urban"].isin(["Urban","Rural"])].copy()
    # counts by group
    def grp_counts(g):
        n = len(g)
        x = g[cover_col].sum()
        p = x / n if n else float("nan")
        # Wilson CI for single proportion
        low, high = proportion_confint(count=x, nobs=n, method="wilson")
        return pd.Series({"n": n, "x": x, "p": p, "p_low": low, "p_high": high})

    tab = dfx.groupby("rural_urban").apply(grp_counts).reset_index()
    # pivot to have Rural/Urban on one row
    row = {"coverage_flag": cover_col}
    for _, r in tab.iterrows():
        g = r["rural_urban"].lower()
        row[f"n_{g}"] = int(r["n"])
        row[f"x_{g}"] = int(r["x"])
        row[f"p_{g}"] = r["p"]
        row[f"p_{g}_low"] = r["p_low"]
        row[f"p_{g}_high"] = r["p_high"]

    # difference & ratio
    p_rur, p_urb = row["p_rural"], row["p_urban"]
    row["risk_difference"] = p_urb - p_rur
    row["risk_ratio"] = (p_urb / p_rur) if (p_rur and not math.isnan(p_rur)) else float("nan")

    # two-proportion z-test (H0: p_rural == p_urban)
    count = np.array([row["x_rural"], row["x_urban"]])
    nobs  = np.array([row["n_rural"], row["n_urban"]])
    z, pval = proportions_ztest(count, nobs, alternative="two-sided")
    row["z_stat"] = z
    row["p_value"] = pval
    return pd.Series(row)

# Build list of coverage columns
coverage_cols = ["covered_any_gi_50mi"] + [
    c for c in zip_universe.columns
    if c.startswith("covered_") and c.endswith("_50mi") and c != "covered_any_gi_50mi"
]

effect_rows = [two_group_stats(zip_universe, col) for col in coverage_cols]
effects_df = pd.DataFrame(effect_rows).sort_values("risk_difference", ascending=False)

# Pretty print & save
pd.set_option("display.max_columns", None)
display(effects_df)
effects_df.to_csv("out/effects_50mi_by_cancer.csv", index=False)
print("Saved out/effects_50mi_by_cancer.csv")


/var/folders/4l/64tth9h948n3dbq1py15y2qr0000gn/T/ipykernel_76260/2799353841.py:16: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  tab = dfx.groupby("rural_urban").apply(grp_counts).reset_index()
/var/folders/4l/64tth9h948n3dbq1py15y2qr0000gn/T/ipykernel_76260/2799353841.py:16: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  tab = dfx.groupby("rural_urban").apply(grp_counts).reset_index()
/var/folders/4l/64tth9h948n3dbq1

,coverage_flag,n_rural,x_rural,p_rural,p_rural_low,p_rural_high,n_urban,x_urban,p_urban,p_urban_low,p_urban_high,risk_difference,risk_ratio,z_stat,p_value
4,covered_hepatocellular_cancer_50mi,16160,5343,0.330631,0.323419,0.337924,16960,13031,0.768337,0.761927,0.774626,0.437706,2.323850,-80.116574,0.0
1,covered_cholangiocarcinoma_50mi,16160,5459,0.337809,0.330557,0.345139,16960,12687,0.748054,0.741465,0.754531,0.410245,2.214427,-74.982950,0.0
5,covered_oesophageal_cancer_50mi,16160,10091,0.624443,0.616948,0.631879,16960,15221,0.897465,0.892809,0.901940,0.273022,1.437224,-58.511680,0.0
6,covered_pancreatic_cancer_50mi,16160,10931,0.676423,0.669169,0.683594,16960,15917,0.938502,0.934786,0.942020,0.262079,1.387448,-60.848818,0.0
3,covered_gastric_cancer_50mi,16160,11332,0.701238,0.694133,0.708246,16960,15995,0.943101,0.939514,0.946488,0.241864,1.344910,-57.916443,0.0
2,covered_colorectal_cancer_50mi,16160,11615,0.718750,0.711767,0.725629,16960,16232,0.957075,0.953920,0.960024,0.238325,1.331583,-59.255837,0.0
0,covered_any_gi_50mi,16160,12500,0.773515,0.766997,0.779903,16960,16472,0.971226,0.968602,0.973638,0.197712,1.255602,-54.337822,0.0


Saved out/effects_50mi_by_cancer.csv
